# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jamieleeuw/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Task type: Scoring / Ranking.

My lane (Refresh / Content Opportunity Scoring) doesn't need a single yes/no label
for each page in isolation — it needs an ORDERED list, because a reviewer only has
capacity to look at the top N pages in a cycle. That's ranking, not plain
classification. Underneath the ranking, though, I'm training a classifier (predicting
P(declining)) and using its predicted probability as the score to sort by — so it's
a classification model used to produce a ranking/scoring output. Not clustering
(I'm not grouping similar pages), and not pure regression (I don't need an exact
number, just a good ordering).

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Target/proxy: is_declining_label = (trend_direction == "down")

This is a PROXY, not a true future outcome. It's a bucket computed from the current
observation window, not something that happened after a decision point. It comes
from a defined rule applied to an observed signal (trend_direction), not from
watching a real future outcome unfold. That's a known limitation I'm inheriting from
the starter dataset — a stronger version of this lane would redefine the label as
a genuine future-window outcome (e.g. decline over the NEXT 30 days, given features
from the PRIOR 90 days), which the lane guide recommends. I'm using the current
proxy for now because it's what the starter data supports, and I'll revisit this
when I have warehouse access.

## 3. Success metric

*One metric you can defend. What number means 'good'?*

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Success metric: Precision@50

I'm picking this over plain accuracy because the output is a ranked queue, not a
label applied to every page — a reviewer will only ever look at the top of the
list (K = however many pages the team has capacity for, e.g. 50). Precision@50
asks: of the top 50 pages the queue recommends, how many are actually declining?
That directly matches how the output gets used. Accuracy across all 30,000 pages
would reward getting the bottom of the list "right" (correctly predicting "not
declining" for obviously fine pages), which nobody cares about — the review team
never sees those rows.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import os

while not os.path.isdir("data/raw") and os.getcwd() != os.path.dirname(os.getcwd()):
    os.chdir("..")

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

print(f"Unit of analysis: one row = one content page")
print(f"Total rows: {len(df):,}")
print(f"Columns: {df.shape[1]}")

# Show a real slice: page identity + the features + the target
cols_to_show = ["content_id", "client_id", "impressions_90d", "days_since_last_update",
                 "avg_position", "ctr", "trend_direction", "is_declining_label"]
df[cols_to_show].head(10)

Unit of analysis: one row = one content page
Total rows: 30,000
Columns: 45


,content_id,client_id,impressions_90d,days_since_last_update,avg_position,ctr,trend_direction,is_declining_label
0,content_304f48230142,client_f369cb89fc,3803,20,10.6,0.76,down,1
1,content_a1fb4e703a9e,client_4e07408562,15320,25,20.3,0.05,down,1
2,content_9aa793d4d895,client_7f2253d7e2,12581,20,36.5,0.09,down,1
3,content_331d6c4de07b,client_19581e27de,11751,22,6.2,0.49,stable,0
4,content_d99b7a2d90ca,client_3fdba35f04,19140,14,44.0,0.13,down,1
5,content_d4084a4bc775,client_f369cb89fc,3970,20,8.5,0.03,down,1
6,content_9a34b442b552,client_8722616204,20,20,7.0,0.00,down,1
7,content_a63219c6e95a,client_19581e27de,1724,22,21.2,0.06,stable,0
8,content_5e6c160719bc,client_6208ef0f77,32574,20,46.0,0.09,down,1
9,content_c27558df2b0c,client_19581e27de,1240,104,4.9,0.16,down,1


One row = one content page. Below is a real slice showing the identity columns,
a few key features, and the target column (is_declining_label) side by side — this
is the shape any model in this lane would actually train on.

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
stale = (df["days_since_last_update"] >= 180)
visible = (df["impressions_90d"] >= 500)
rule_flag_pct = (stale & visible).mean()

print(f"Share of pages flagged by the hand rule (stale AND visible): {rule_flag_pct:.3f}")
print(f"Actual declining rate in the full dataset: {df['is_declining_label'].mean():.3f}")

Share of pages flagged by the hand rule (stale AND visible): 0.001
Actual declining rate in the full dataset: 0.542


A fixed if-statement rule (stale AND visible) only flags about 0.1% of pages —
far too narrow to be a useful review queue, even though 54.2% of all pages are
actually declining. The rule misses almost everything, because "declining" doesn't
reduce to one or two clean threshold checks — it's a pattern spread across many
correlated, noisy signals (position, CTR, age, freshness, volume) with no single
obvious cutoff. That's exactly the kind of messy, multi-variable pattern a learned
model is suited to pick up and a hand rule isn't.

This isn't just a hunch — the verified pipeline results (client-holdout validated)
show a real gap: the hand-written baseline scores Precision@50 = 0.240, while a
random forest trained on the same data scores Precision@50 = 0.740. That's not an
in-sample artifact; it held up under proper client-holdout evaluation.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.